# Notebook 25 — Expanding a custom gene set with regulatory evidence

`pathway_subtyping.genesets.RegulatoryGeneSetExpander` suggests
candidate additions to a seed set based on shared regulatory signal.

This is a **human-in-the-loop** tool — candidates are *suggestions*,
not auto-expansions. The expander flags candidates that are already
listed in curated databases so you can prioritise novel ones.

Research use only. Not for clinical decision-making.

In [ ]:
import numpy as np
import pandas as pd
from pathway_subtyping.genesets import (
    CoexpressionBackend, RegulatoryGeneSetExpander,
)

rng = np.random.default_rng(0)
# Synthetic cohort: 10 genes co-expressed, 10 independent.
n_samples = 200
latent = rng.standard_normal(n_samples)
df = pd.DataFrame({
    **{f'PATH_{i}': latent + rng.normal(0, 0.1, n_samples) for i in range(10)},
    **{f'OTHER_{i}': rng.standard_normal(n_samples) for i in range(10)},
})
seed = ['PATH_0', 'PATH_1', 'PATH_2']

## 1. Expand the seed set

`CoexpressionBackend` is the default fallback — Pearson-correlation
based. `BorzoiBackend` is the production substitute when the
`[genesets]` extra is installed.

In [ ]:
expander = RegulatoryGeneSetExpander(backend=CoexpressionBackend(df))
result = expander.expand(
    seed_genes=seed,
    candidate_genes=df.columns.tolist(),
    top_n=10,
    known_members={'reactome': ['PATH_5', 'PATH_7']},  # illustrative
)
result.as_dataframe()

## 2. Explain each suggestion

`source_genes` on each candidate lists the seed genes that
contributed most to its score — a quick sanity-check for the user.

In [ ]:
for candidate in result.top(5):
    print(f"{candidate.gene:15s} score={candidate.score:.3f} "
          f"in_curated_db={candidate.in_curated_db} "
          f"top_sources={candidate.source_genes}")

## 3. Recall against a held-out ground truth (acceptance proxy)

The roadmap criterion is that on a held-out Reactome pathway, the
top-20 suggestions contain at least 30% of the held-out pathway
members. `recall_against` is the one-line check.

In [ ]:
held_out = [f'PATH_{i}' for i in range(3, 10)]
print(f'recall@10: {result.recall_against(held_out):.2f}')

## Further reading

- Linder J et al. (2024). *Borzoi: predicting RNA-seq coverage from
  DNA sequence with improved accuracy for distal regulation.*
- PSF v0.6 roadmap — Phase 2 F7:
  [docs/roadmap-v06-codeberg.md](../../docs/roadmap-v06-codeberg.md)